# 04. 출시 후 분석: 패치 및 운영 방향성 제안 LLM

이 노트북은 `03_run_llm_allgames_analysi.ipynb`, `03-1`, `03-2`의 구조를 출시 후 분석용으로 바꾼 버전이다.

## 목적

특정 게임 1개를 지정한 뒤 Steam 리뷰를 LLM으로 분석하고, 그 결과를 바탕으로 패치 및 운영 방향성을 제안한다.

## 핵심 흐름

```text
특정 게임 리뷰 선택
↓
영어 리뷰 / Steam 구매 / 무료 수령 제외 / 얼리액세스 제외 / 의미 있는 리뷰 필터링
↓
전체 리뷰 + 최근 90일/30일 여부 플래그 생성
↓
LLM 리뷰 분석
감정, 주요 이슈, 세부 이슈 태그, urgency 후보, 개선 제안 분류
↓
이슈별 집계
영향 리뷰 수, 부정/혼합 리뷰 수, High urgency 리뷰 수, 최근 리뷰 수, 초기 이탈 리뷰 수
↓
규칙 기반 대응 구분 확정
즉시 확인 / 단기 개선 / 장기 검토 / 검토 필요 / 계속 살릴 강점
↓
LLM 패치·운영 제안 생성
LLM은 우선순위를 새로 판단하지 않고, 이미 계산된 집계와 대응 구분을 문장화한다.
```

## 중요한 원칙

- `llm_urgency_candidate`는 LLM이 리뷰 문맥을 보고 분류한 보조 신호다.
- 최종 대응 구분은 LLM에게 맡기지 않고, 아래 집계값과 규칙으로 계산한다.
- 마지막 LLM은 수치와 대응 구분을 바꾸지 않고, 패치·운영 제안을 보고서 문장으로 정리하는 역할만 한다.

# 0. 환경 설정

In [ ]:
# ============================================================
# 기본 라이브러리
# ============================================================
import os
import re
import ast
import json
import time
import asyncio
import platform
from pathlib import Path
from typing import List, Literal, Optional
from datetime import datetime

# ============================================================
# 데이터 분석용 라이브러리
# ============================================================
import pandas as pd
import numpy as np
from IPython.display import display, Markdown

# ============================================================
# LLM / 환경변수 / 스키마 관련 라이브러리
# ============================================================
from dotenv import load_dotenv
from pydantic import BaseModel, Field
from tqdm.auto import tqdm

from pydantic_ai import Agent
from pydantic_ai.models.google import GoogleModel, GoogleModelSettings
from pydantic_ai.providers.google import GoogleProvider

# ============================================================
# 출력 옵션
# ============================================================
pd.set_option("display.max_columns", 160)
pd.set_option("display.max_colwidth", 180)

print("라이브러리 로드 완료")

# 1. 경로 및 실행 설정

In [ ]:
# ============================================================
# 프로젝트 루트 설정
# ============================================================
# 다른 환경에서 실행할 경우 ROOT만 본인 프로젝트 경로에 맞게 수정한다.
ROOT = Path.cwd()

# Jupyter 실행 위치가 하위 폴더일 수 있으므로 data 폴더가 보일 때까지 상위 폴더를 탐색한다.
if not (ROOT / "data").exists():
    for parent in ROOT.parents:
        if (parent / "data").exists():
            ROOT = parent
            break

load_dotenv()
load_dotenv(ROOT / ".env")

# ============================================================
# 파일 탐색 함수
# ============================================================
def find_existing_file(filename, extra_candidates=None):
    """프로젝트 내부에서 파일 위치가 조금 달라도 찾을 수 있게 한다."""
    candidates = []
    if extra_candidates:
        candidates.extend([Path(p) for p in extra_candidates])

    candidates.extend([
        ROOT / filename,
        ROOT / "data" / filename,
        ROOT / "data" / "raw" / filename,
        ROOT / "data" / "preprocessed" / filename,
        ROOT / "data" / "outputs" / filename,
        Path("/mnt/data") / filename,
    ])

    for path in candidates:
        if path.exists():
            return path
    return None

# ============================================================
# 입력 파일 경로
# ============================================================
# 1순위: 01 전처리 노트북에서 만든 LLM 후보 리뷰 파일
PREPROCESSED_REVIEWS_PATH = ROOT / "data" / "outputs" / "preprocess_llm" / "llm_preprocessed_reviews.csv"

# 2순위: 원천 CSV에서 직접 구성
RAW_REVIEWS_PATH = find_existing_file("steam_indie_reviews.csv")
RAW_GAMES_PATH = find_existing_file("steam_indie_games.csv")
RAW_SUMMARY_PATH = find_existing_file("steam_indie_review_summary.csv")

# ============================================================
# 출력 폴더
# ============================================================
RUN_NAME = "postlaunch_zoonomaly_patch_ops_v1"
OUTPUT_DIR = ROOT / "data" / "outputs" / RUN_NAME
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

LLM_INPUT_PATH = OUTPUT_DIR / "postlaunch_llm_input_reviews.csv"
FILTER_LOG_PATH = OUTPUT_DIR / "postlaunch_filter_log.csv"
SAMPLE_SUMMARY_PATH = OUTPUT_DIR / "postlaunch_sample_summary.csv"

CHECKPOINT_PATH = OUTPUT_DIR / "postlaunch_llm_review_analysis_checkpoint.json"
RESULT_JSON_PATH = OUTPUT_DIR / "postlaunch_llm_review_analysis_result.json"
RESULT_CSV_PATH = OUTPUT_DIR / "postlaunch_llm_review_analysis_result.csv"
ISSUE_TAG_FLAT_PATH = OUTPUT_DIR / "postlaunch_llm_issue_tags_flat.csv"

GAME_OVERVIEW_PATH = OUTPUT_DIR / "postlaunch_game_overview.csv"
ISSUE_SUMMARY_PATH = OUTPUT_DIR / "postlaunch_issue_patch_summary.csv"
RESPONSE_GROUP_SUMMARY_PATH = OUTPUT_DIR / "postlaunch_response_group_summary.csv"
PATCH_EVIDENCE_PATH = OUTPUT_DIR / "postlaunch_patch_evidence_base.csv"
TABLEAU_SOURCE_PATH = OUTPUT_DIR / "postlaunch_tableau_issue_source.csv"

PATCH_PLAN_JSON_PATH = OUTPUT_DIR / "postlaunch_patch_operation_plan.json"
PATCH_PLAN_CSV_PATH = OUTPUT_DIR / "postlaunch_patch_operation_plan_tasks.csv"
PATCH_PLAN_MD_PATH = OUTPUT_DIR / "postlaunch_patch_operation_plan.md"

print("ROOT:", ROOT)
print("전처리 리뷰 파일:", PREPROCESSED_REVIEWS_PATH, PREPROCESSED_REVIEWS_PATH.exists())
print("원천 리뷰 파일:", RAW_REVIEWS_PATH)
print("게임 메타 파일:", RAW_GAMES_PATH)
print("결과 저장 폴더:", OUTPUT_DIR)

In [ ]:
# ============================================================
# 실행 여부
# ============================================================
RUN_LLM = True
RESET_CHECKPOINT = False
RUN_PATCH_PLAN_LLM = True

# ============================================================
# 분석 대상 게임
# ============================================================
TARGET_APPID = 2101890
TARGET_GAME_NAME = "Zoonomaly"

# ============================================================
# 리뷰 기간 설정
# ============================================================
# all: 조건에 맞는 전체 리뷰를 분석하되, 최근 90일/30일 여부를 플래그로 남긴다.
# recent_90d: 최근 90일 리뷰만 LLM 분석한다.
# recent_30d: 최근 30일 리뷰만 LLM 분석한다.
ANALYSIS_REVIEW_WINDOW = "all"

# 최근 리뷰 판정 기준
# dataset_max: 현재 CSV 안에서 가장 최신 리뷰일을 기준으로 최근 N일을 계산한다.
# today: 오늘 날짜를 기준으로 최근 N일을 계산한다.
REFERENCE_DATE_MODE = "dataset_max"
RECENT_DAYS_MAIN = 90
RECENT_DAYS_SUB = 30

# ============================================================
# 리뷰 필터 설정
# ============================================================
LANGUAGES = ["english"]
STEAM_PURCHASE_ONLY = True
EXCLUDE_RECEIVED_FOR_FREE = True
EXCLUDE_EARLY_ACCESS_REVIEWS = True
MEANINGFUL_REVIEW_ONLY = True

MIN_REVIEW_LEN = 20
MIN_WORD_COUNT = 4
MAX_REVIEW_CHARS = 1500

# ============================================================
# 샘플링 설정
# ============================================================
# 출시 후 분석은 문제 진단이 목적이므로 부정 리뷰를 조금 더 많이 보되,
# 강점 유지를 위해 긍정 리뷰도 반드시 일부 포함한다.
MAX_TOTAL_REVIEWS = 1000
SAMPLE_MODE = "negative_recent_priority"
NEGATIVE_SAMPLE_RATIO = 0.70
RANDOM_STATE = 42

# ============================================================
# LLM 호출 설정
# ============================================================
BATCH_SIZE = 3
MAX_CONCURRENT = 1
MAX_RETRIES = 3
REQUEST_SLEEP_SEC = 1
CHUNK_SIZE = MAX_CONCURRENT * 3
SAVE_RESULT_JSON = True

# ============================================================
# 패치/운영 대응 구분 규칙 설정
# ============================================================
HIGH_URGENCY_RATIO_CUTOFF = 40.0
MIN_HIGH_URGENCY_COUNT = 2
MIN_NEGATIVE_MIXED_COUNT = 3
MIN_RECENT_ISSUE_COUNT = 1
EARLY_CHURN_HOURS = 1.0
EARLY_FRICTION_HOURS = 5.0

print("분석 대상:", TARGET_GAME_NAME, TARGET_APPID)
print("분석 리뷰 범위:", ANALYSIS_REVIEW_WINDOW)
print("RUN_LLM:", RUN_LLM)
print("RUN_PATCH_PLAN_LLM:", RUN_PATCH_PLAN_LLM)

# 2. 공통 함수

In [ ]:
# ============================================================
# JSON/CSV 저장 보조 함수
# ============================================================
def to_serializable(obj):
    """Pydantic, pandas, numpy 값을 JSON 저장 가능한 기본 타입으로 바꾼다."""
    if isinstance(obj, BaseModel):
        return to_serializable(obj.model_dump())
    if isinstance(obj, dict):
        return {k: to_serializable(v) for k, v in obj.items()}
    if isinstance(obj, list):
        return [to_serializable(v) for v in obj]
    if isinstance(obj, tuple):
        return [to_serializable(v) for v in obj]
    if isinstance(obj, np.integer):
        return int(obj)
    if isinstance(obj, np.floating):
        return None if np.isnan(obj) else float(obj)
    if isinstance(obj, np.bool_):
        return bool(obj)
    if isinstance(obj, pd.Timestamp):
        return None if pd.isna(obj) else obj.isoformat()
    if isinstance(obj, datetime):
        return obj.isoformat()
    try:
        if pd.isna(obj):
            return None
    except Exception:
        pass
    return obj


def safe_json_dumps(obj):
    return json.dumps(to_serializable(obj), ensure_ascii=False)


def parse_issue_tags(value):
    """llm_issue_tags 값을 안정적으로 list[dict] 형태로 변환한다."""
    if value is None:
        return []
    if isinstance(value, float) and pd.isna(value):
        return []
    if isinstance(value, list):
        tags = value
    elif isinstance(value, str):
        text = value.strip()
        if not text or text.lower() in ["nan", "none", "null"]:
            return []
        try:
            tags = json.loads(text)
        except Exception:
            try:
                tags = ast.literal_eval(text)
            except Exception:
                return []
    else:
        return []

    if not isinstance(tags, list):
        return []

    normalized = []
    for tag in tags:
        if isinstance(tag, BaseModel):
            tag = tag.model_dump()
        if isinstance(tag, dict):
            normalized.append({
                "category": tag.get("category"),
                "sentiment": tag.get("sentiment"),
                "evidence": tag.get("evidence"),
            })
    return normalized


def ensure_columns(df, columns):
    out = df.copy()
    for col in columns:
        if col not in out.columns:
            out[col] = pd.NA
    return out[columns].copy()


def save_csv_safely(df, path, columns=None, json_cols=None, encoding="utf-8-sig"):
    path = Path(path)
    path.parent.mkdir(parents=True, exist_ok=True)

    if df is None:
        df = pd.DataFrame()
    out = df.copy()

    if columns is not None:
        out = ensure_columns(out, columns)

    for col in (json_cols or []):
        if col in out.columns:
            out[col] = out[col].apply(lambda x: safe_json_dumps(parse_issue_tags(x)))

    out.to_csv(path, index=False, encoding=encoding)
    return out


# ============================================================
# 텍스트/날짜 보조 함수
# ============================================================
def clean_review_text(text, max_chars=MAX_REVIEW_CHARS):
    if pd.isna(text):
        return ""
    text = str(text)
    text = re.sub(r"\s+", " ", text).strip()
    return text[:max_chars]


def count_words(text):
    if pd.isna(text):
        return 0
    return len(str(text).split())


def make_release_period(days):
    if pd.isna(days):
        return "unknown"
    days = float(days)
    if days < 0:
        return "before_release"
    if days <= 7:
        return "D0-D7"
    if days <= 30:
        return "D8-D30"
    if days <= 90:
        return "D31-D90"
    if days <= 180:
        return "D91-D180"
    if days <= 365:
        return "D181-D365"
    return "D366+"


def make_playtime_stage(hours):
    if pd.isna(hours):
        return "unknown"
    hours = float(hours)
    if hours < 1:
        return "0-1h"
    if hours < 5:
        return "1-5h"
    if hours < 20:
        return "5-20h"
    if hours < 50:
        return "20-50h"
    return "50h+"


def print_filter_step(log_rows, step, before, after):
    removed = before - after
    removed_rate = removed / before if before else 0
    log_rows.append({
        "step": step,
        "before_rows": before,
        "after_rows": after,
        "removed_rows": removed,
        "removed_rate": removed_rate,
    })
    print(f"{step}: {before:,} -> {after:,} / 제거 {removed:,} ({removed_rate:.2%})")


# ============================================================
# checkpoint 함수
# ============================================================
def load_checkpoint(path=CHECKPOINT_PATH):
    if path.exists():
        with open(path, "r", encoding="utf-8") as f:
            return json.load(f)
    return []


def save_checkpoint(results, path=CHECKPOINT_PATH):
    safe_results = to_serializable(results)
    path.parent.mkdir(parents=True, exist_ok=True)
    with open(path, "w", encoding="utf-8") as f:
        json.dump(safe_results, f, ensure_ascii=False, indent=2)


def load_existing_results():
    if RESULT_JSON_PATH.exists():
        with open(RESULT_JSON_PATH, "r", encoding="utf-8") as f:
            return json.load(f)
    return load_checkpoint(CHECKPOINT_PATH)


def extract_usage_tokens(result):
    input_tokens = 0
    output_tokens = 0
    try:
        usage = result.usage()
        input_tokens = (
            getattr(usage, "input_tokens", None)
            or getattr(usage, "request_tokens", None)
            or getattr(usage, "prompt_tokens", None)
            or 0
        )
        output_tokens = (
            getattr(usage, "output_tokens", None)
            or getattr(usage, "response_tokens", None)
            or getattr(usage, "completion_tokens", None)
            or 0
        )
    except Exception:
        pass
    return int(input_tokens or 0), int(output_tokens or 0)

# 3. Vertex AI / PydanticAI 설정

In [ ]:
# ============================================================
# Vertex AI 설정
# ============================================================
GOOGLE_CLOUD_PROJECT = (
    os.getenv("GOOGLE_CLOUD_PROJECT")
    or os.getenv("VERTEX_PROJECT_ID")
    or os.getenv("GCP_PROJECT_ID")
)

GOOGLE_CLOUD_LOCATION = (
    os.getenv("GOOGLE_CLOUD_LOCATION")
    or os.getenv("VERTEX_LOCATION")
    or "global"
)

GEMINI_MODEL = os.getenv("GEMINI_MODEL", "gemini-3.1-flash-lite-preview")

if GOOGLE_CLOUD_PROJECT:
    vertex_provider = GoogleProvider(
        vertexai=True,
        project=GOOGLE_CLOUD_PROJECT,
        location=GOOGLE_CLOUD_LOCATION,
    )

    vertex_model = GoogleModel(
        GEMINI_MODEL,
        provider=vertex_provider,
    )

    print("Vertex AI project:", GOOGLE_CLOUD_PROJECT)
    print("Vertex AI location:", GOOGLE_CLOUD_LOCATION)
    print("Gemini model:", GEMINI_MODEL)
    print("Vertex 모델 생성: O")
else:
    vertex_provider = None
    vertex_model = None

    print("Vertex AI project: 미설정")
    print("Vertex 모델 생성: X")
    print("실제 LLM 실행 전 .env에 GOOGLE_CLOUD_PROJECT를 설정하세요.")

review_settings = GoogleModelSettings(temperature=0.0)
patch_plan_settings = GoogleModelSettings(temperature=0.0)

# 4. 리뷰 데이터 로드 및 출시 후 분석용 전처리

In [ ]:
# ============================================================
# 데이터 로드
# ============================================================
# 1순위: 전처리 산출물 사용
# 2순위: 원천 리뷰 + 게임 메타데이터를 직접 결합

if PREPROCESSED_REVIEWS_PATH.exists():
    review_source_type = "preprocessed_llm_candidate"
    reviews = pd.read_csv(PREPROCESSED_REVIEWS_PATH)
else:
    review_source_type = "raw_reviews"
    if RAW_REVIEWS_PATH is None:
        raise FileNotFoundError("steam_indie_reviews.csv를 찾지 못했습니다.")
    reviews = pd.read_csv(RAW_REVIEWS_PATH)

print("리뷰 소스:", review_source_type)
print("원본 리뷰 수:", len(reviews))
print("원본 컬럼 수:", len(reviews.columns))

# 게임 메타데이터 결합
if RAW_GAMES_PATH is not None:
    games = pd.read_csv(RAW_GAMES_PATH)
    game_meta_cols = [
        col for col in [
            "appid", "name", "release_date", "genres", "categories", "tags", "price",
            "developers", "publishers", "total_reviews", "positive", "negative"
        ]
        if col in games.columns
    ]
    game_meta = games[game_meta_cols].drop_duplicates("appid")
else:
    games = pd.DataFrame()
    game_meta = pd.DataFrame(columns=["appid"])

# 리뷰 데이터에 게임 메타가 없거나 부족할 수 있으므로 appid 기준으로 보강한다.
if "appid" in reviews.columns and len(game_meta) > 0:
    rename_meta = {}
    if "name" in game_meta.columns and "game_name" not in reviews.columns:
        rename_meta["name"] = "game_name"
    if "genres" in game_meta.columns and "genres_text" not in reviews.columns:
        rename_meta["genres"] = "genres_text"
    if "categories" in game_meta.columns and "categories_text" not in reviews.columns:
        rename_meta["categories"] = "categories_text"
    if "tags" in game_meta.columns and "top_steam_tags_text" not in reviews.columns:
        rename_meta["tags"] = "top_steam_tags_text"

    game_meta_for_merge = game_meta.rename(columns=rename_meta).copy()

    # release_date는 양쪽에 있을 수 있으므로, 리뷰 데이터에 없을 때만 붙인다.
    merge_cols = ["appid"]
    for col in game_meta_for_merge.columns:
        if col == "appid":
            continue
        if col not in reviews.columns:
            merge_cols.append(col)

    reviews = reviews.merge(game_meta_for_merge[merge_cols], on="appid", how="left")

print("메타데이터 결합 후 컬럼 수:", len(reviews.columns))
display(reviews.head(2))

In [ ]:
# ============================================================
# 컬럼 표준화
# ============================================================
df_all = reviews.copy()

# recommendationid 문자화
if "recommendationid" not in df_all.columns and "review_id" in df_all.columns:
    df_all["recommendationid"] = df_all["review_id"]

df_all["recommendationid"] = df_all["recommendationid"].astype(str)
df_all["appid"] = df_all["appid"].astype(int)

# 게임명
if "game_name" not in df_all.columns:
    df_all["game_name"] = TARGET_GAME_NAME

# 리뷰 작성일
if "review_datetime" in df_all.columns:
    df_all["review_datetime"] = pd.to_datetime(df_all["review_datetime"], errors="coerce")
elif "created_date" in df_all.columns:
    df_all["review_datetime"] = pd.to_datetime(df_all["created_date"], errors="coerce")
elif "timestamp_created" in df_all.columns:
    df_all["review_datetime"] = pd.to_datetime(df_all["timestamp_created"], unit="s", errors="coerce")
else:
    df_all["review_datetime"] = pd.NaT

# 출시일
if "release_date" in df_all.columns:
    df_all["release_date"] = pd.to_datetime(df_all["release_date"], errors="coerce")
else:
    df_all["release_date"] = pd.NaT

# 리뷰 텍스트
if "review_text_for_llm" not in df_all.columns:
    if "review" in df_all.columns:
        df_all["review_text_for_llm"] = df_all["review"].apply(clean_review_text)
    elif "review_text" in df_all.columns:
        df_all["review_text_for_llm"] = df_all["review_text"].apply(clean_review_text)
    else:
        df_all["review_text_for_llm"] = ""
else:
    df_all["review_text_for_llm"] = df_all["review_text_for_llm"].apply(clean_review_text)

# 리뷰 길이/단어 수
if "review_len" not in df_all.columns:
    df_all["review_len"] = df_all["review_text_for_llm"].str.len()

df_all["review_word_count"] = df_all["review_text_for_llm"].apply(count_words)

# Steam 라벨
if "steam_label_text" not in df_all.columns:
    if "voted_up" in df_all.columns:
        df_all["steam_label_text"] = np.where(df_all["voted_up"].astype(bool), "positive", "negative")
    else:
        df_all["steam_label_text"] = "unknown"

# 플레이타임
if "playtime_at_review_hours" not in df_all.columns:
    if "playtime_at_review" in df_all.columns:
        df_all["playtime_at_review_hours"] = pd.to_numeric(df_all["playtime_at_review"], errors="coerce") / 60
    else:
        df_all["playtime_at_review_hours"] = np.nan

df_all["playtime_at_review_hours"] = pd.to_numeric(df_all["playtime_at_review_hours"], errors="coerce")
df_all["playtime_stage"] = df_all["playtime_at_review_hours"].apply(make_playtime_stage)

# 출시 후 경과일/구간
if "days_from_release" not in df_all.columns:
    df_all["days_from_release"] = (df_all["review_datetime"] - df_all["release_date"]).dt.days

df_all["release_period"] = df_all["days_from_release"].apply(make_release_period)

# 메타 문자열 보강
for col in ["genres_text", "categories_text", "top_steam_tags_text"]:
    if col not in df_all.columns:
        df_all[col] = ""
    df_all[col] = df_all[col].fillna("").astype(str)

# 의미 있는 리뷰 여부
if "is_meaningful_review" not in df_all.columns:
    df_all["is_meaningful_review"] = (
        (df_all["review_len"] >= MIN_REVIEW_LEN)
        & (df_all["review_word_count"] >= MIN_WORD_COUNT)
    )

# 최근 기준일
if REFERENCE_DATE_MODE == "today":
    reference_date = pd.Timestamp.today().normalize()
else:
    reference_date = df_all["review_datetime"].max()
    if pd.isna(reference_date):
        reference_date = pd.Timestamp.today().normalize()

reference_date = pd.Timestamp(reference_date).normalize()
df_all["review_age_days"] = (reference_date - df_all["review_datetime"]).dt.days

df_all["is_recent_90d"] = df_all["review_age_days"].between(0, RECENT_DAYS_MAIN, inclusive="both")
df_all["is_recent_30d"] = df_all["review_age_days"].between(0, RECENT_DAYS_SUB, inclusive="both")
df_all["is_launch_30d"] = df_all["days_from_release"].between(0, 30, inclusive="both")

def make_review_recency_bucket(age_days):
    if pd.isna(age_days):
        return "unknown"
    if age_days <= RECENT_DAYS_SUB:
        return f"recent_{RECENT_DAYS_SUB}d"
    if age_days <= RECENT_DAYS_MAIN:
        return f"recent_{RECENT_DAYS_MAIN}d"
    return "older"

df_all["review_recency_bucket"] = df_all["review_age_days"].apply(make_review_recency_bucket)

print("기준일:", reference_date.date())
print("표준화 완료")
print("전체 리뷰 수:", len(df_all))
display(df_all[["recommendationid", "appid", "game_name", "review_datetime", "release_date", "steam_label_text", "release_period", "playtime_stage", "is_recent_90d"]].head())

# 5. 분석 대상 게임 필터링 및 샘플링

In [ ]:
# ============================================================
# 분석 조건 필터링
# ============================================================
filtered = df_all.copy()
log_rows = []

# 1. 게임 지정
before = len(filtered)
if TARGET_APPID is not None:
    filtered = filtered[filtered["appid"] == int(TARGET_APPID)]
elif TARGET_GAME_NAME:
    filtered = filtered[filtered["game_name"].fillna("").str.contains(TARGET_GAME_NAME, case=False, na=False)]
print_filter_step(log_rows, "대상 게임 필터", before, len(filtered))

# 2. 언어
if LANGUAGES and "language" in filtered.columns:
    before = len(filtered)
    filtered = filtered[filtered["language"].isin(LANGUAGES)]
    print_filter_step(log_rows, "언어 필터", before, len(filtered))

# 3. Steam 구매 리뷰
if STEAM_PURCHASE_ONLY and "steam_purchase" in filtered.columns:
    before = len(filtered)
    filtered = filtered[filtered["steam_purchase"] == True]
    print_filter_step(log_rows, "Steam 구매 리뷰 필터", before, len(filtered))

# 4. 무료 수령 제외
if EXCLUDE_RECEIVED_FOR_FREE and "received_for_free" in filtered.columns:
    before = len(filtered)
    filtered = filtered[filtered["received_for_free"] == False]
    print_filter_step(log_rows, "무료 수령 리뷰 제외", before, len(filtered))

# 5. 얼리액세스 리뷰 제외
if EXCLUDE_EARLY_ACCESS_REVIEWS and "written_during_early_access" in filtered.columns:
    before = len(filtered)
    filtered = filtered[filtered["written_during_early_access"] == False]
    print_filter_step(log_rows, "얼리액세스 작성 리뷰 제외", before, len(filtered))

# 6. 의미 있는 리뷰만 사용
if MEANINGFUL_REVIEW_ONLY and "is_meaningful_review" in filtered.columns:
    before = len(filtered)
    filtered = filtered[filtered["is_meaningful_review"] == True]
    print_filter_step(log_rows, "의미 있는 리뷰 필터", before, len(filtered))

# 7. 분석 리뷰 기간
before = len(filtered)
if ANALYSIS_REVIEW_WINDOW == "recent_90d":
    filtered = filtered[filtered["is_recent_90d"] == True]
elif ANALYSIS_REVIEW_WINDOW == "recent_30d":
    filtered = filtered[filtered["is_recent_30d"] == True]
elif ANALYSIS_REVIEW_WINDOW == "all":
    pass
else:
    raise ValueError("ANALYSIS_REVIEW_WINDOW은 all, recent_90d, recent_30d 중 하나여야 합니다.")
print_filter_step(log_rows, f"분석 리뷰 기간 필터: {ANALYSIS_REVIEW_WINDOW}", before, len(filtered))

filter_log = pd.DataFrame(log_rows)
filter_log.to_csv(FILTER_LOG_PATH, index=False, encoding="utf-8-sig")

print("최종 후보 리뷰 수:", len(filtered))
print("최종 후보 게임 수:", filtered["appid"].nunique())
display(filter_log)

In [ ]:
# ============================================================
# 출시 후 분석용 샘플링
# ============================================================
# - 전체 후보 리뷰가 MAX_TOTAL_REVIEWS 이하이면 모두 사용한다.
# - 초과하면 부정 리뷰를 우선 확보하고, 최근 리뷰와 유용함 투표를 참고해 샘플링한다.

SORT_COLS = ["is_recent_90d", "is_recent_30d", "review_datetime", "votes_up", "weighted_vote_score"]
for col in SORT_COLS:
    if col not in filtered.columns:
        filtered[col] = 0


def pick_top_or_sample(group_df, n, random_state=RANDOM_STATE):
    if n <= 0 or len(group_df) == 0:
        return group_df.iloc[0:0].copy()
    if len(group_df) <= n:
        return group_df.copy()

    # 최근 리뷰와 영향력이 높은 리뷰를 먼저 남긴다.
    sort_cols = [col for col in ["is_recent_90d", "is_recent_30d", "review_datetime", "votes_up", "weighted_vote_score"] if col in group_df.columns]
    return (
        group_df
        .sort_values(sort_cols, ascending=[False, False, False, False, False][:len(sort_cols)])
        .head(n)
        .copy()
    )


def sample_postlaunch_reviews(df):
    if len(df) <= MAX_TOTAL_REVIEWS:
        return df.sort_values("review_datetime", ascending=False).copy()

    negative_target = int(MAX_TOTAL_REVIEWS * NEGATIVE_SAMPLE_RATIO)
    positive_target = MAX_TOTAL_REVIEWS - negative_target

    neg = df[df["steam_label_text"] == "negative"].copy()
    pos = df[df["steam_label_text"] == "positive"].copy()
    other = df[~df["steam_label_text"].isin(["positive", "negative"])].copy()

    neg_sample = pick_top_or_sample(neg, negative_target)
    remaining_slots = MAX_TOTAL_REVIEWS - len(neg_sample)
    pos_sample = pick_top_or_sample(pos, remaining_slots)

    remaining_slots = MAX_TOTAL_REVIEWS - len(neg_sample) - len(pos_sample)
    other_sample = pick_top_or_sample(other, remaining_slots)

    sampled = pd.concat([neg_sample, pos_sample, other_sample], ignore_index=True)

    # 혹시 남은 슬롯이 있으면 나머지에서 채운다.
    if len(sampled) < MAX_TOTAL_REVIEWS:
        used_ids = set(sampled["recommendationid"].astype(str))
        remain = df[~df["recommendationid"].astype(str).isin(used_ids)].copy()
        fill = pick_top_or_sample(remain, MAX_TOTAL_REVIEWS - len(sampled))
        sampled = pd.concat([sampled, fill], ignore_index=True)

    return sampled.drop_duplicates("recommendationid").sort_values("review_datetime", ascending=False).copy()


df_llm_input = sample_postlaunch_reviews(filtered)

# LLM 입력 컬럼 정리
LLM_INPUT_COLUMNS = [
    "recommendationid", "appid", "game_name", "language",
    "review_datetime", "release_date", "days_from_release", "release_period",
    "review_age_days", "review_recency_bucket", "is_recent_90d", "is_recent_30d", "is_launch_30d",
    "steam_label_text", "voted_up",
    "playtime_at_review_hours", "playtime_stage",
    "votes_up", "weighted_vote_score",
    "steam_purchase", "received_for_free", "written_during_early_access",
    "genres_text", "categories_text", "top_steam_tags_text",
    "review_text_for_llm", "review_len", "review_word_count", "is_meaningful_review",
]

df_llm_input = ensure_columns(df_llm_input, LLM_INPUT_COLUMNS)
df_llm_input.to_csv(LLM_INPUT_PATH, index=False, encoding="utf-8-sig")

sample_summary = pd.DataFrame({
    "항목": [
        "대상 게임", "appid", "후보 리뷰 수", "LLM 입력 리뷰 수", "분석 범위",
        "최근 90일 리뷰 수", "최근 30일 리뷰 수", "출시 초기 30일 리뷰 수",
        "Steam 긍정 리뷰 수", "Steam 부정 리뷰 수",
        "평균 플레이타임", "중앙값 플레이타임",
    ],
    "값": [
        TARGET_GAME_NAME, TARGET_APPID, len(filtered), len(df_llm_input), ANALYSIS_REVIEW_WINDOW,
        int(df_llm_input["is_recent_90d"].sum()), int(df_llm_input["is_recent_30d"].sum()), int(df_llm_input["is_launch_30d"].sum()),
        int((df_llm_input["steam_label_text"] == "positive").sum()),
        int((df_llm_input["steam_label_text"] == "negative").sum()),
        round(df_llm_input["playtime_at_review_hours"].mean(), 2),
        round(df_llm_input["playtime_at_review_hours"].median(), 2),
    ]
})

sample_summary.to_csv(SAMPLE_SUMMARY_PATH, index=False, encoding="utf-8-sig")

print("LLM 입력 리뷰 저장:", LLM_INPUT_PATH)
print("LLM 입력 리뷰 수:", len(df_llm_input))
display(sample_summary)
display(df_llm_input.head(3))

# 6. LLM 리뷰 분석 출력 스키마

In [ ]:
# ============================================================
# 이슈 카테고리
# ============================================================
IssueCategory = Literal[
    "bug",
    "optimization",
    "performance",
    "crash",
    "control",
    "balance",
    "difficulty",
    "content_volume",
    "story",
    "translation_localization",
    "ui_ux",
    "price_value",
    "multiplayer_network",
    "save_progression",
    "graphics_audio",
    "gameplay_loop",
    "monetization",
    "developer_communication",
    "positive_praise",
    "progression_grind",
    "other",
]


class IssueTag(BaseModel):
    category: IssueCategory = Field(description="리뷰에서 언급된 세부 이슈 카테고리")
    sentiment: Literal["positive", "negative", "neutral", "mixed"] = Field(description="이 이슈에 대한 감정 방향")
    evidence: str = Field(description="원문 리뷰를 바탕으로 한 짧은 판단 근거", min_length=1, max_length=180)


class SteamPostlaunchReviewAnalysis(BaseModel):
    recommendationid: str = Field(description="입력 리뷰 ID 그대로 반환")

    llm_sentiment: Literal["positive", "negative", "neutral", "mixed"] = Field(
        description="리뷰 본문 기준 전체 감정"
    )
    sentiment_score: int = Field(ge=1, le=5, description="1=매우 부정, 3=중립, 5=매우 긍정")

    llm_primary_issue: IssueCategory = Field(description="리뷰의 대표 이슈")
    llm_issue_tags: List[IssueTag] = Field(default_factory=list, description="리뷰에서 발견된 세부 이슈 목록")

    llm_urgency_candidate: Literal["low", "medium", "high"] = Field(
        description="LLM이 리뷰 문맥 기준으로 판단한 시급도 후보. 최종 우선순위가 아니라 보조 신호"
    )

    llm_patch_scope: Literal["hotfix", "minor_patch", "major_update", "communication", "keep_monitoring"] = Field(
        description="리뷰 1개 기준으로 보이는 대응 범위 후보"
    )

    llm_player_impact: Literal[
        "progress_blocker",
        "early_churn_risk",
        "experience_friction",
        "content_longevity",
        "positive_strength",
        "unclear",
    ] = Field(description="해당 리뷰가 시사하는 플레이어 영향 유형")

    llm_review_summary: str = Field(description="리뷰 핵심 내용 요약", min_length=5, max_length=240)
    llm_suggested_action: str = Field(description="개발사 또는 운영자가 참고할 수 있는 개선 방향", min_length=5, max_length=300)


class BatchSteamPostlaunchReviewAnalysis(BaseModel):
    results: List[SteamPostlaunchReviewAnalysis] = Field(description="리뷰별 분석 결과 목록")


ISSUE_KR_MAP = {
    "bug": "버그",
    "optimization": "최적화",
    "performance": "성능",
    "crash": "크래시",
    "control": "조작감",
    "balance": "밸런스",
    "difficulty": "난이도",
    "content_volume": "콘텐츠 분량",
    "story": "스토리",
    "translation_localization": "번역/현지화",
    "ui_ux": "UI/UX",
    "price_value": "가격/가치",
    "multiplayer_network": "멀티/네트워크",
    "save_progression": "저장/진행",
    "graphics_audio": "그래픽/사운드",
    "gameplay_loop": "게임플레이 루프",
    "monetization": "과금",
    "developer_communication": "개발사 소통",
    "positive_praise": "긍정 칭찬",
    "progression_grind": "성장/반복 노가다",
    "other": "기타",
}

# 7. LLM 리뷰 분석 프롬프트 및 Agent

In [ ]:
# ============================================================
# 리뷰 분석 system prompt
# ============================================================
review_system_prompt = """
당신은 Steam 인디게임 출시 후 리뷰를 분석하는 데이터 분석 보조자입니다.

각 리뷰에 대해 다음을 분류/작성하세요.
1. 리뷰 본문 기준 감정(llm_sentiment)
2. 가장 핵심적인 이슈(llm_primary_issue)
3. 세부 이슈 태그(llm_issue_tags)
4. 리뷰 문맥상 개선 시급도 후보(llm_urgency_candidate)
5. 리뷰 1개 기준 대응 범위 후보(llm_patch_scope)
6. 플레이어 영향 유형(llm_player_impact)
7. 리뷰 요약(llm_review_summary)
8. 개발사 관점 개선 제안(llm_suggested_action)

중요한 제한 사항:
- recommendationid는 반드시 입력값 그대로 반환하세요.
- voted_up과 Steam 라벨은 참고 정보일 뿐, 감정은 review 본문 기준으로 판단하세요.
- llm_urgency_candidate는 최종 패치 우선순위가 아니라 리뷰 문맥상 문제 강도 후보입니다.
- 최종 대응 구분은 후속 코드에서 이슈 반복 수, 최근 리뷰 반복 여부, 부정/혼합 리뷰 수, High urgency 리뷰 수를 기준으로 별도 계산합니다.
- 리뷰 본문에 근거가 없는 문제를 새로 상상하지 마세요.
- 게임 메타데이터와 플레이타임은 맥락 참고용입니다. 리뷰 본문에 없는 내용을 억지로 추론하지 마세요.
- 긍정 리뷰는 유지해야 할 강점으로 해석할 수 있게 positive_praise 또는 실제 칭찬받은 이슈를 태그로 남기세요.
- 부정 리뷰는 개발사가 실제로 확인할 수 있는 수준으로 구체적인 suggested_action을 작성하세요.
- 매우 짧거나 농담/밈 위주 리뷰는 과잉 해석하지 말고 other, low, keep_monitoring에 가깝게 보수적으로 분류하세요.
"""

if vertex_model is not None:
    review_agent = Agent(
        vertex_model,
        output_type=BatchSteamPostlaunchReviewAnalysis,
        system_prompt=review_system_prompt,
        retries=MAX_RETRIES,
        output_retries=3,
    )
else:
    review_agent = None

print("리뷰 분석 Agent 생성 여부:", "O" if review_agent is not None else "X")

In [ ]:
def build_review_batch_prompt(batch_df):
    """여러 리뷰를 한 번에 LLM에게 보내기 위한 프롬프트 생성 함수."""
    blocks = []

    for _, row in batch_df.iterrows():
        block = f"""
[REVIEW]
recommendationid: {row["recommendationid"]}
appid: {row["appid"]}
game_name: {row.get("game_name", "")}
genres: {row.get("genres_text", "")}
categories: {row.get("categories_text", "")}
steam_top_tags: {row.get("top_steam_tags_text", "")}
release_date: {row.get("release_date", "")}
review_datetime: {row.get("review_datetime", "")}
days_from_release: {row.get("days_from_release", "")}
release_period: {row.get("release_period", "")}
review_age_days: {row.get("review_age_days", "")}
review_recency_bucket: {row.get("review_recency_bucket", "")}
is_recent_90d: {row.get("is_recent_90d", "")}
is_recent_30d: {row.get("is_recent_30d", "")}
is_launch_30d: {row.get("is_launch_30d", "")}
language: {row.get("language", "")}
voted_up: {row.get("voted_up", "")}
steam_label_text: {row.get("steam_label_text", "")}
playtime_at_review_hours: {row.get("playtime_at_review_hours", "")}
playtime_stage: {row.get("playtime_stage", "")}
votes_up: {row.get("votes_up", "")}
weighted_vote_score: {row.get("weighted_vote_score", "")}

review:
{row.get("review_text_for_llm", "")}
[/REVIEW]
"""
        blocks.append(block)

    prompt = (
        f"다음 {len(batch_df)}개의 Steam 리뷰를 각각 분석해주세요.\n"
        "반드시 입력된 recommendationid를 그대로 유지해서 반환하세요.\n"
        "결과는 지정된 Pydantic 스키마에 맞게 반환하세요.\n\n"
        + "\n".join(blocks)
    )
    return prompt

# 8. LLM 리뷰 분석 실행

In [ ]:
sem = asyncio.Semaphore(MAX_CONCURRENT)


async def analyze_review_batch(batch_df, all_results, stats, pbar):
    """리뷰 배치 1개를 PydanticAI + Vertex AI Gemini로 분석한다."""
    async with sem:
        prompt = build_review_batch_prompt(batch_df)

        for attempt in range(MAX_RETRIES):
            try:
                if review_agent is None:
                    raise RuntimeError(
                        "review_agent가 생성되지 않았습니다. .env의 GOOGLE_CLOUD_PROJECT, gcloud ADC 인증, pydantic-ai 설치 여부를 확인하세요."
                    )

                result = await review_agent.run(prompt, model_settings=review_settings)
                output = result.output
                output_items = getattr(output, "results", [])

                input_tokens, output_tokens = extract_usage_tokens(result)
                stats["input_tokens"] += input_tokens
                stats["output_tokens"] += output_tokens
                stats["requests"] += 1

                input_ids = set(batch_df["recommendationid"].astype(str).tolist())
                matched_ids = set()

                for item in output_items:
                    rid = str(item.recommendationid)
                    if rid not in input_ids:
                        continue

                    row = batch_df[batch_df["recommendationid"].astype(str) == rid].iloc[0]
                    matched_ids.add(rid)

                    record = {
                        "analysis_status": "success",
                        "recommendationid": rid,
                        "appid": row.get("appid"),
                        "game_name": row.get("game_name", ""),
                        "review_datetime": row.get("review_datetime", None),
                        "release_date": row.get("release_date", None),
                        "days_from_release": row.get("days_from_release", None),
                        "release_period": row.get("release_period", None),
                        "review_age_days": row.get("review_age_days", None),
                        "review_recency_bucket": row.get("review_recency_bucket", None),
                        "is_recent_90d": row.get("is_recent_90d", None),
                        "is_recent_30d": row.get("is_recent_30d", None),
                        "is_launch_30d": row.get("is_launch_30d", None),
                        "steam_label_text": row.get("steam_label_text", ""),
                        "playtime_at_review_hours": row.get("playtime_at_review_hours", None),
                        "playtime_stage": row.get("playtime_stage", None),
                        "votes_up": row.get("votes_up", None),
                        "weighted_vote_score": row.get("weighted_vote_score", None),
                        "llm_sentiment": item.llm_sentiment,
                        "sentiment_score": item.sentiment_score,
                        "llm_primary_issue": item.llm_primary_issue,
                        "llm_issue_tags": [tag.model_dump() for tag in item.llm_issue_tags],
                        "llm_urgency_candidate": item.llm_urgency_candidate,
                        "llm_patch_scope": item.llm_patch_scope,
                        "llm_player_impact": item.llm_player_impact,
                        "llm_review_summary": item.llm_review_summary,
                        "llm_suggested_action": item.llm_suggested_action,
                        "error_message": None,
                    }
                    all_results.append(to_serializable(record))

                # LLM 결과에서 누락된 리뷰는 missing으로 남긴다.
                missing_ids = input_ids - matched_ids
                for rid in missing_ids:
                    row = batch_df[batch_df["recommendationid"].astype(str) == rid].iloc[0]
                    all_results.append({
                        "analysis_status": "missing_in_llm_output",
                        "recommendationid": rid,
                        "appid": row.get("appid"),
                        "game_name": row.get("game_name", ""),
                        "review_datetime": row.get("review_datetime", None),
                        "release_date": row.get("release_date", None),
                        "days_from_release": row.get("days_from_release", None),
                        "release_period": row.get("release_period", None),
                        "review_age_days": row.get("review_age_days", None),
                        "review_recency_bucket": row.get("review_recency_bucket", None),
                        "is_recent_90d": row.get("is_recent_90d", None),
                        "is_recent_30d": row.get("is_recent_30d", None),
                        "is_launch_30d": row.get("is_launch_30d", None),
                        "steam_label_text": row.get("steam_label_text", ""),
                        "playtime_at_review_hours": row.get("playtime_at_review_hours", None),
                        "playtime_stage": row.get("playtime_stage", None),
                        "votes_up": row.get("votes_up", None),
                        "weighted_vote_score": row.get("weighted_vote_score", None),
                        "llm_sentiment": None,
                        "sentiment_score": None,
                        "llm_primary_issue": None,
                        "llm_issue_tags": [],
                        "llm_urgency_candidate": None,
                        "llm_patch_scope": None,
                        "llm_player_impact": None,
                        "llm_review_summary": None,
                        "llm_suggested_action": None,
                        "error_message": "LLM output에 해당 recommendationid가 없음",
                    })

                save_checkpoint(all_results)
                pbar.update(len(batch_df))
                return

            except Exception as e:
                if attempt < MAX_RETRIES - 1:
                    time.sleep(2 ** attempt)
                    continue

                for _, row in batch_df.iterrows():
                    all_results.append({
                        "analysis_status": "failed",
                        "recommendationid": str(row.get("recommendationid")),
                        "appid": row.get("appid"),
                        "game_name": row.get("game_name", ""),
                        "review_datetime": row.get("review_datetime", None),
                        "release_date": row.get("release_date", None),
                        "days_from_release": row.get("days_from_release", None),
                        "release_period": row.get("release_period", None),
                        "review_age_days": row.get("review_age_days", None),
                        "review_recency_bucket": row.get("review_recency_bucket", None),
                        "is_recent_90d": row.get("is_recent_90d", None),
                        "is_recent_30d": row.get("is_recent_30d", None),
                        "is_launch_30d": row.get("is_launch_30d", None),
                        "steam_label_text": row.get("steam_label_text", ""),
                        "playtime_at_review_hours": row.get("playtime_at_review_hours", None),
                        "playtime_stage": row.get("playtime_stage", None),
                        "votes_up": row.get("votes_up", None),
                        "weighted_vote_score": row.get("weighted_vote_score", None),
                        "llm_sentiment": None,
                        "sentiment_score": None,
                        "llm_primary_issue": None,
                        "llm_issue_tags": [],
                        "llm_urgency_candidate": None,
                        "llm_patch_scope": None,
                        "llm_player_impact": None,
                        "llm_review_summary": None,
                        "llm_suggested_action": None,
                        "error_message": str(e),
                    })
                save_checkpoint(all_results)
                pbar.update(len(batch_df))
                return


async def run_review_analysis(df):
    """전체 LLM 리뷰 분석 실행."""
    if RESET_CHECKPOINT and CHECKPOINT_PATH.exists():
        CHECKPOINT_PATH.unlink()
        print("기존 checkpoint 삭제:", CHECKPOINT_PATH)

    all_results = list(load_checkpoint())

    target_ids = set(df["recommendationid"].astype(str))
    all_results = [row for row in all_results if str(row.get("recommendationid")) in target_ids]

    done_ids = {
        str(row.get("recommendationid"))
        for row in all_results
        if row.get("analysis_status") in ["success", "missing_in_llm_output", "failed"]
    }

    to_process = df[~df["recommendationid"].astype(str).isin(done_ids)].copy()

    stats = {
        "input_tokens": 0,
        "output_tokens": 0,
        "requests": 0,
        "checkpoint_count": len(done_ids),
        "to_process_count": len(to_process),
    }

    if len(to_process) == 0:
        print("새로 처리할 리뷰가 없습니다. checkpoint 또는 기존 결과를 사용합니다.")
        return all_results, stats

    batches = [to_process.iloc[i:i + BATCH_SIZE] for i in range(0, len(to_process), BATCH_SIZE)]

    with tqdm(total=len(to_process), desc="출시 후 리뷰 LLM 분석 진행") as pbar:
        for start in range(0, len(batches), CHUNK_SIZE):
            chunk = batches[start:start + CHUNK_SIZE]
            tasks = [analyze_review_batch(batch_df, all_results, stats, pbar) for batch_df in chunk]
            await asyncio.gather(*tasks)

            if REQUEST_SLEEP_SEC > 0:
                await asyncio.sleep(REQUEST_SLEEP_SEC)

    return all_results, stats

In [ ]:
# ============================================================
# LLM 리뷰 분석 실행
# ============================================================
if RUN_LLM:
    all_results, review_llm_stats = await run_review_analysis(df_llm_input)
else:
    all_results = load_existing_results()
    review_llm_stats = {
        "input_tokens": 0,
        "output_tokens": 0,
        "requests": 0,
        "checkpoint_count": len(all_results),
        "to_process_count": 0,
    }

print("LLM 리뷰 분석 결과 수:", len(all_results))
print(review_llm_stats)

# 9. 리뷰 분석 결과 저장

In [ ]:
def classify_sentiment_relation(row):
    steam_label = row.get("steam_label_text")
    llm_sentiment = row.get("llm_sentiment")

    if steam_label == "positive":
        if llm_sentiment == "positive":
            return "exact_match"
        if llm_sentiment == "mixed":
            return "partial_match"
        if llm_sentiment == "negative":
            return "mismatch"
        return "unclear"

    if steam_label == "negative":
        if llm_sentiment == "negative":
            return "exact_match"
        if llm_sentiment == "mixed":
            return "partial_match"
        if llm_sentiment == "positive":
            return "mismatch"
        return "unclear"

    return "unknown"


REVIEW_RESULT_COLUMNS = [
    "analysis_status",
    "recommendationid", "appid", "game_name",
    "review_datetime", "release_date", "days_from_release", "release_period",
    "review_age_days", "review_recency_bucket", "is_recent_90d", "is_recent_30d", "is_launch_30d",
    "steam_label_text", "playtime_at_review_hours", "playtime_stage", "votes_up", "weighted_vote_score",
    "llm_sentiment", "sentiment_score", "llm_primary_issue", "llm_issue_tags",
    "llm_urgency_candidate", "llm_patch_scope", "llm_player_impact",
    "llm_review_summary", "llm_suggested_action",
    "steam_llm_sentiment_relation",
]

if len(all_results) > 0:
    df_result_all = pd.DataFrame(all_results)
    df_result_all = ensure_columns(df_result_all, list(dict.fromkeys(REVIEW_RESULT_COLUMNS + ["error_message"])))

    df_result_all["recommendationid"] = df_result_all["recommendationid"].astype(str)
    for col in ["review_datetime", "release_date"]:
        df_result_all[col] = pd.to_datetime(df_result_all[col], errors="coerce")

    for col in ["is_recent_90d", "is_recent_30d", "is_launch_30d"]:
        df_result_all[col] = df_result_all[col].astype("boolean")

    df_result_all["llm_issue_tags"] = df_result_all["llm_issue_tags"].apply(parse_issue_tags)
    df_result_all["steam_llm_sentiment_relation"] = df_result_all.apply(classify_sentiment_relation, axis=1)

    df_result = df_result_all[df_result_all["analysis_status"] == "success"].copy()
    df_failed_log = df_result_all[df_result_all["analysis_status"] != "success"].copy()
else:
    df_result_all = pd.DataFrame(columns=list(dict.fromkeys(REVIEW_RESULT_COLUMNS + ["error_message"])))
    df_result = pd.DataFrame(columns=REVIEW_RESULT_COLUMNS)
    df_failed_log = pd.DataFrame(columns=list(dict.fromkeys(REVIEW_RESULT_COLUMNS + ["error_message"])))

# CSV 저장
save_csv_safely(
    df_result,
    RESULT_CSV_PATH,
    columns=REVIEW_RESULT_COLUMNS,
    json_cols=["llm_issue_tags"],
)

if SAVE_RESULT_JSON:
    with open(RESULT_JSON_PATH, "w", encoding="utf-8") as f:
        json.dump(to_serializable(df_result.to_dict(orient="records")), f, ensure_ascii=False, indent=2)

print("리뷰 단위 결과 CSV 저장:", RESULT_CSV_PATH)
print("성공 분석 리뷰 수:", len(df_result))
print("실패/누락 리뷰 수:", len(df_failed_log))
display(df_result.head(3))

In [ ]:
# ============================================================
# 이슈 태그 펼치기
# ============================================================
ISSUE_TAG_FLAT_COLUMNS = [
    "recommendationid", "appid", "game_name",
    "steam_label_text", "llm_sentiment", "llm_primary_issue", "llm_urgency_candidate",
    "llm_patch_scope", "llm_player_impact",
    "release_period", "review_age_days", "review_recency_bucket", "is_recent_90d", "is_recent_30d", "is_launch_30d",
    "playtime_at_review_hours", "playtime_stage", "votes_up", "weighted_vote_score",
    "llm_issue_category", "issue_name_kor", "llm_issue_sentiment", "llm_issue_evidence",
]


def flatten_issue_tags(df):
    flat_rows = []

    if len(df) == 0 or "llm_issue_tags" not in df.columns:
        return pd.DataFrame(columns=ISSUE_TAG_FLAT_COLUMNS)

    for _, row in df.iterrows():
        tags = parse_issue_tags(row.get("llm_issue_tags", []))

        for tag in tags:
            category = tag.get("category")
            issue_name_kor = ISSUE_KR_MAP.get(category, category)

            flat_rows.append({
                "recommendationid": row.get("recommendationid"),
                "appid": row.get("appid"),
                "game_name": row.get("game_name"),
                "steam_label_text": row.get("steam_label_text"),
                "llm_sentiment": row.get("llm_sentiment"),
                "llm_primary_issue": row.get("llm_primary_issue"),
                "llm_urgency_candidate": row.get("llm_urgency_candidate"),
                "llm_patch_scope": row.get("llm_patch_scope"),
                "llm_player_impact": row.get("llm_player_impact"),
                "release_period": row.get("release_period"),
                "review_age_days": row.get("review_age_days"),
                "review_recency_bucket": row.get("review_recency_bucket"),
                "is_recent_90d": row.get("is_recent_90d"),
                "is_recent_30d": row.get("is_recent_30d"),
                "is_launch_30d": row.get("is_launch_30d"),
                "playtime_at_review_hours": row.get("playtime_at_review_hours"),
                "playtime_stage": row.get("playtime_stage"),
                "votes_up": row.get("votes_up"),
                "weighted_vote_score": row.get("weighted_vote_score"),
                "llm_issue_category": category,
                "issue_name_kor": issue_name_kor,
                "llm_issue_sentiment": tag.get("sentiment"),
                "llm_issue_evidence": tag.get("evidence"),
            })

    return pd.DataFrame(flat_rows, columns=ISSUE_TAG_FLAT_COLUMNS)


df_issue_tags_flat = flatten_issue_tags(df_result)
df_issue_tags_flat.to_csv(ISSUE_TAG_FLAT_PATH, index=False, encoding="utf-8-sig")

print("이슈 태그 펼친 결과 저장:", ISSUE_TAG_FLAT_PATH)
print("이슈 태그 행 수:", len(df_issue_tags_flat))
display(df_issue_tags_flat.head())

# 10. 게임 현황 요약

In [ ]:
# ============================================================
# 게임 현황 요약
# ============================================================
if len(df_result) == 0:
    raise ValueError("성공한 LLM 분석 결과가 없습니다. 앞 단계 실행 결과를 확인하세요.")

game_overview = pd.DataFrame({
    "항목": [
        "게임명", "appid", "분석 리뷰 수", "분석 태그 수",
        "분석 표본 내 Steam 라벨 긍정률", "LLM 긍정률", "LLM 부정+혼합 비율",
        "High urgency 비율", "최근 90일 리뷰 수", "최근 30일 리뷰 수", "출시 초기 30일 리뷰 수",
        "분석 리뷰 시작일", "분석 리뷰 종료일",
        "평균 플레이타임(리뷰 시점)", "중앙값 플레이타임(리뷰 시점)",
    ],
    "값": [
        df_result["game_name"].dropna().iloc[0] if df_result["game_name"].notna().any() else TARGET_GAME_NAME,
        int(df_result["appid"].dropna().iloc[0]) if df_result["appid"].notna().any() else TARGET_APPID,
        len(df_result),
        len(df_issue_tags_flat),
        round((df_result["steam_label_text"].eq("positive").mean() * 100), 1),
        round((df_result["llm_sentiment"].eq("positive").mean() * 100), 1),
        round((df_result["llm_sentiment"].isin(["negative", "mixed"]).mean() * 100), 1),
        round((df_result["llm_urgency_candidate"].eq("high").mean() * 100), 1),
        int(df_result["is_recent_90d"].fillna(False).sum()),
        int(df_result["is_recent_30d"].fillna(False).sum()),
        int(df_result["is_launch_30d"].fillna(False).sum()),
        df_result["review_datetime"].min(),
        df_result["review_datetime"].max(),
        round(df_result["playtime_at_review_hours"].mean(), 2),
        round(df_result["playtime_at_review_hours"].median(), 2),
    ]
})

game_overview.to_csv(GAME_OVERVIEW_PATH, index=False, encoding="utf-8-sig")

print("게임 현황 요약 저장:", GAME_OVERVIEW_PATH)
display(game_overview)

# 11. 이슈별 집계 및 규칙 기반 대응 구분

In [ ]:
# ============================================================
# 이슈별 집계용 기본 테이블
# ============================================================
# 같은 리뷰 안에서 같은 이슈가 중복으로 들어간 경우 1번만 본다.
tag_base = df_issue_tags_flat.dropna(subset=["recommendationid", "llm_issue_category"]).copy()
tag_base["recommendationid"] = tag_base["recommendationid"].astype(str)

tag_base = tag_base.drop_duplicates(["recommendationid", "llm_issue_category"])

# 분석용 플래그
tag_base["is_llm_positive"] = tag_base["llm_sentiment"].eq("positive")
tag_base["is_llm_negative"] = tag_base["llm_sentiment"].eq("negative")
tag_base["is_llm_mixed"] = tag_base["llm_sentiment"].eq("mixed")
tag_base["is_negative_or_mixed"] = tag_base["llm_sentiment"].isin(["negative", "mixed"])
tag_base["is_high_urgency"] = tag_base["llm_urgency_candidate"].eq("high")
tag_base["is_recent_90d"] = tag_base["is_recent_90d"].fillna(False).astype(bool)
tag_base["is_recent_30d"] = tag_base["is_recent_30d"].fillna(False).astype(bool)
tag_base["is_launch_30d"] = tag_base["is_launch_30d"].fillna(False).astype(bool)

tag_base["is_early_churn_negative"] = (
    tag_base["is_negative_or_mixed"]
    & (pd.to_numeric(tag_base["playtime_at_review_hours"], errors="coerce") <= EARLY_CHURN_HOURS)
)

tag_base["is_early_friction_negative"] = (
    tag_base["is_negative_or_mixed"]
    & (pd.to_numeric(tag_base["playtime_at_review_hours"], errors="coerce") <= EARLY_FRICTION_HOURS)
)

print("리뷰-이슈 단위 행 수:", len(tag_base))
display(tag_base.head())

In [ ]:
# ============================================================
# 대표 근거 문장 추출 함수
# ============================================================
def collect_top_evidence(group, max_items=3):
    # 부정/혼합 + high urgency + 최근 리뷰를 우선 보여준다.
    temp = group.copy()
    temp["sort_high"] = temp["is_high_urgency"].astype(int)
    temp["sort_negative"] = temp["is_negative_or_mixed"].astype(int)
    temp["sort_recent"] = temp["is_recent_90d"].astype(int)
    temp["sort_votes"] = pd.to_numeric(temp["votes_up"], errors="coerce").fillna(0)

    temp = temp.sort_values(
        ["sort_high", "sort_negative", "sort_recent", "sort_votes"],
        ascending=[False, False, False, False],
    )

    evidences = []
    for text in temp["llm_issue_evidence"].dropna().astype(str):
        text = text.strip()
        if text and text not in evidences:
            evidences.append(text)
        if len(evidences) >= max_items:
            break

    return " / ".join(evidences)


# ============================================================
# 이슈별 요약표 생성
# ============================================================
issue_summary = (
    tag_base
    .groupby(["llm_issue_category", "issue_name_kor"], dropna=False)
    .agg(
        issue_review_count=("recommendationid", "nunique"),
        positive_review_count=("is_llm_positive", "sum"),
        negative_review_count=("is_llm_negative", "sum"),
        mixed_review_count=("is_llm_mixed", "sum"),
        negative_mixed_review_count=("is_negative_or_mixed", "sum"),
        high_urgency_review_count=("is_high_urgency", "sum"),
        recent_90d_review_count=("is_recent_90d", "sum"),
        recent_30d_review_count=("is_recent_30d", "sum"),
        launch_30d_review_count=("is_launch_30d", "sum"),
        early_churn_review_count=("is_early_churn_negative", "sum"),
        early_friction_review_count=("is_early_friction_negative", "sum"),
        avg_playtime_hours=("playtime_at_review_hours", "mean"),
        avg_votes_up=("votes_up", "mean"),
    )
    .reset_index()
)

issue_summary["high_urgency_ratio"] = (
    issue_summary["high_urgency_review_count"] / issue_summary["issue_review_count"] * 100
).round(1)

issue_summary["negative_mixed_ratio"] = (
    issue_summary["negative_mixed_review_count"] / issue_summary["issue_review_count"] * 100
).round(1)

# 대표 근거 문장 결합
evidence_df = (
    tag_base
    .groupby(["llm_issue_category", "issue_name_kor"], dropna=False)
    .apply(lambda g: collect_top_evidence(g, max_items=3), include_groups=False)
    .reset_index(name="representative_evidence")
)

issue_summary = issue_summary.merge(evidence_df, on=["llm_issue_category", "issue_name_kor"], how="left")

print("이슈별 요약 행 수:", len(issue_summary))
display(issue_summary.sort_values("issue_review_count", ascending=False).head(10))

In [ ]:
# ============================================================
# 규칙 기반 대응 구분
# ============================================================
# LLM이 최종 우선순위를 판단하지 않도록, 대응 구분은 아래 규칙으로 고정한다.

IMMEDIATE_CATEGORIES = {"bug", "crash", "save_progression", "performance", "optimization"}
SHORT_TERM_CATEGORIES = {"gameplay_loop", "control", "ui_ux", "balance", "difficulty", "progression_grind"}
LONG_TERM_CATEGORIES = {"content_volume", "story", "multiplayer_network", "developer_communication", "monetization"}
REVIEW_NEEDED_CATEGORIES = {"graphics_audio", "translation_localization", "price_value", "other"}


def classify_response_group(row):
    category = row["llm_issue_category"]
    issue_count = row["issue_review_count"]
    neg_mixed = row["negative_mixed_review_count"]
    high_count = row["high_urgency_review_count"]
    high_ratio = row["high_urgency_ratio"]
    recent_90 = row["recent_90d_review_count"]
    early_churn = row["early_churn_review_count"]
    positive_count = row["positive_review_count"]

    # 긍정 중심 이슈는 강점 유지로 분리한다.
    if category == "positive_praise" or (positive_count > neg_mixed and high_count == 0):
        return "계속 살릴 강점"

    # 플레이를 막는 기술 문제는 즉시 확인 대상으로 둔다.
    if category in IMMEDIATE_CATEGORIES and (
        high_count >= MIN_HIGH_URGENCY_COUNT
        or high_ratio >= HIGH_URGENCY_RATIO_CUTOFF
        or neg_mixed >= MIN_NEGATIVE_MIXED_COUNT
        or recent_90 >= MIN_RECENT_ISSUE_COUNT
    ):
        return "즉시 확인"

    # 경험 품질과 초반 이탈에 연결되는 문제는 단기 개선 대상으로 둔다.
    if category in SHORT_TERM_CATEGORIES and (
        neg_mixed >= MIN_NEGATIVE_MIXED_COUNT
        or high_count >= MIN_HIGH_URGENCY_COUNT
        or early_churn >= 1
        or issue_count >= MIN_NEGATIVE_MIXED_COUNT
    ):
        return "단기 개선"

    # 개발 범위가 큰 항목은 장기 검토로 둔다.
    if category in LONG_TERM_CATEGORIES:
        return "장기 검토"

    # 증거가 애매하거나 보조 품질 항목은 검토 필요로 둔다.
    if category in REVIEW_NEEDED_CATEGORIES:
        return "검토 필요"

    # 나머지는 규모와 시급도에 따라 보수적으로 분류한다.
    if high_count >= MIN_HIGH_URGENCY_COUNT or high_ratio >= HIGH_URGENCY_RATIO_CUTOFF:
        return "즉시 확인"
    if neg_mixed >= MIN_NEGATIVE_MIXED_COUNT:
        return "단기 개선"
    return "검토 필요"


def make_rule_basis(row):
    return (
        f"영향 리뷰 {int(row['issue_review_count'])}개, "
        f"부정/혼합 {int(row['negative_mixed_review_count'])}개, "
        f"High urgency {int(row['high_urgency_review_count'])}개({row['high_urgency_ratio']}%), "
        f"최근 90일 {int(row['recent_90d_review_count'])}개, "
        f"초기 이탈 후보 {int(row['early_churn_review_count'])}개"
    )


response_order_map = {
    "즉시 확인": 1,
    "단기 개선": 2,
    "장기 검토": 3,
    "검토 필요": 4,
    "계속 살릴 강점": 5,
}

issue_summary["response_group"] = issue_summary.apply(classify_response_group, axis=1)
issue_summary["response_order"] = issue_summary["response_group"].map(response_order_map)
issue_summary["rule_basis"] = issue_summary.apply(make_rule_basis, axis=1)

issue_summary = issue_summary.sort_values(
    ["response_order", "high_urgency_review_count", "negative_mixed_review_count", "issue_review_count", "recent_90d_review_count"],
    ascending=[True, False, False, False, False],
).reset_index(drop=True)

issue_summary.to_csv(ISSUE_SUMMARY_PATH, index=False, encoding="utf-8-sig")

print("이슈별 패치 요약 저장:", ISSUE_SUMMARY_PATH)
display(issue_summary.head(15))

# 12. 대응 유형별 운영 제안 근거표

In [ ]:
# ============================================================
# 대응 유형별 요약
# ============================================================
response_group_summary = (
    issue_summary
    .groupby("response_group", dropna=False)
    .agg(
        issue_count=("llm_issue_category", "nunique"),
        issue_names=("issue_name_kor", lambda x: ", ".join(x.astype(str).tolist())),
        total_issue_reviews=("issue_review_count", "sum"),
        total_negative_mixed_reviews=("negative_mixed_review_count", "sum"),
        total_high_urgency_reviews=("high_urgency_review_count", "sum"),
        total_recent_90d_reviews=("recent_90d_review_count", "sum"),
        total_early_churn_reviews=("early_churn_review_count", "sum"),
    )
    .reset_index()
)

response_group_summary["response_order"] = response_group_summary["response_group"].map(response_order_map)
response_group_summary = response_group_summary.sort_values("response_order").reset_index(drop=True)

# 보고서용 제안 방향 기본 문장
response_direction_map = {
    "즉시 확인": "플레이 방해 요소, 저장 문제, 버그, 성능 문제를 먼저 재현·확인한다.",
    "단기 개선": "반복 플레이 구조, 난이도, 조작감, UI/UX처럼 경험 품질을 낮추는 요소를 개선한다.",
    "장기 검토": "콘텐츠 확장, 멀티플레이, 스토리 보강처럼 업데이트 로드맵 수준에서 검토한다.",
    "검토 필요": "리뷰 원문과 태그 근거를 추가 확인한 뒤 대응 방향을 정한다.",
    "계속 살릴 강점": "긍정적으로 평가된 요소는 유지하고 패치 과정에서 훼손되지 않게 관리한다.",
}

response_group_summary["제안 방향"] = response_group_summary["response_group"].map(response_direction_map)
response_group_summary.to_csv(RESPONSE_GROUP_SUMMARY_PATH, index=False, encoding="utf-8-sig")

print("대응 유형별 요약 저장:", RESPONSE_GROUP_SUMMARY_PATH)
display(response_group_summary)

In [ ]:
# ============================================================
# LLM 패치/운영 제안용 근거 데이터 생성
# ============================================================
# 너무 많은 이슈를 LLM에게 넘기지 않도록 대응 구분별 상위 이슈만 사용한다.
TOP_N_ISSUES_PER_GROUP = 5

patch_evidence_base = (
    issue_summary
    .sort_values(
        ["response_order", "high_urgency_review_count", "negative_mixed_review_count", "issue_review_count", "recent_90d_review_count"],
        ascending=[True, False, False, False, False],
    )
    .groupby("response_group", group_keys=False)
    .head(TOP_N_ISSUES_PER_GROUP)
    .reset_index(drop=True)
)

patch_evidence_cols = [
    "response_group", "response_order", "llm_issue_category", "issue_name_kor",
    "issue_review_count", "positive_review_count", "negative_review_count", "mixed_review_count", "negative_mixed_review_count",
    "high_urgency_review_count", "high_urgency_ratio", "negative_mixed_ratio",
    "recent_90d_review_count", "recent_30d_review_count", "launch_30d_review_count",
    "early_churn_review_count", "early_friction_review_count",
    "rule_basis", "representative_evidence",
]

patch_evidence_base = ensure_columns(patch_evidence_base, patch_evidence_cols)
patch_evidence_base.to_csv(PATCH_EVIDENCE_PATH, index=False, encoding="utf-8-sig")

print("패치/운영 제안 근거 저장:", PATCH_EVIDENCE_PATH)
display(patch_evidence_base)

In [ ]:
# ============================================================
# Tableau용 원천 데이터 생성
# ============================================================
# 1행 = 리뷰-이슈 단위
# 이슈별/기간별/플레이타임별 필터링이 가능하도록 저장한다.

tableau_source = tag_base.merge(
    issue_summary[["llm_issue_category", "response_group", "response_order", "rule_basis"]],
    on="llm_issue_category",
    how="left",
)

tableau_source.to_csv(TABLEAU_SOURCE_PATH, index=False, encoding="utf-8-sig")

print("Tableau용 출시 후 이슈 원천 데이터 저장:", TABLEAU_SOURCE_PATH)
print("데이터 크기:", tableau_source.shape)
display(tableau_source.head())

# 13. 패치·운영 제안 LLM 생성

In [ ]:
# ============================================================
# 패치/운영 제안 출력 스키마
# ============================================================
class PatchTask(BaseModel):
    response_group: Literal["즉시 확인", "단기 개선", "장기 검토", "검토 필요", "계속 살릴 강점"] = Field(
        description="코드에서 계산한 대응 구분 그대로 사용"
    )
    issue_name_kor: str = Field(description="이슈명")
    evidence_summary: str = Field(description="제공된 집계 근거를 바탕으로 한 요약")
    recommended_action: str = Field(description="개발사 관점의 구체적인 패치 또는 운영 액션")
    expected_effect: str = Field(description="이 액션으로 기대할 수 있는 효과")
    caution: str = Field(description="실행 시 주의할 점")


class PostlaunchOperationPlan(BaseModel):
    one_line_summary: str = Field(description="출시 후 분석 결과 한 줄 요약")
    check_now: List[str] = Field(description="지금 먼저 확인할 일")
    next_update: List[str] = Field(description="다음 업데이트에서 개선할 일")
    long_term: List[str] = Field(description="장기적으로 보강할 일")
    keep_strength: List[str] = Field(description="계속 살릴 강점")
    patch_tasks: List[PatchTask] = Field(description="이슈별 패치/운영 작업 목록")
    report_text: str = Field(description="보고서에 바로 넣을 수 있는 설명 문단")

In [ ]:
def df_to_records_for_prompt(df, max_rows=30):
    temp = df.head(max_rows).copy()
    return json.dumps(to_serializable(temp.to_dict(orient="records")), ensure_ascii=False, indent=2)


def build_patch_plan_prompt(game_overview, response_group_summary, patch_evidence_base):
    overview_json = df_to_records_for_prompt(game_overview, max_rows=50)
    group_json = df_to_records_for_prompt(response_group_summary, max_rows=20)
    evidence_json = df_to_records_for_prompt(patch_evidence_base, max_rows=30)

    prompt = f"""
아래는 특정 Steam 인디게임의 출시 후 리뷰 LLM 분석 결과를 집계한 데이터입니다.

[게임 현황 요약]
{overview_json}

[대응 유형별 집계]
{group_json}

[이슈별 패치/운영 근거]
{evidence_json}

작성 규칙:
1. response_group은 이미 코드에서 계산된 값입니다. 절대 새로 바꾸지 마세요.
2. 우선순위나 수치를 새로 만들지 마세요.
3. 제공된 issue_review_count, negative_mixed_review_count, high_urgency_review_count, recent_90d_review_count, early_churn_review_count를 근거로만 작성하세요.
4. llm_urgency_candidate는 리뷰 문맥상 보조 신호이며 실제 장애 지표가 아닙니다. 문장에서도 과장하지 마세요.
5. 추천 액션은 인디게임 개발사가 다음 패치/운영 회의에서 바로 검토할 수 있는 표현으로 작성하세요.
6. 리뷰에 근거가 없는 기능 추가나 대규모 시스템 개편을 단정하지 마세요.
7. 출력은 지정된 Pydantic 스키마에 맞게 작성하세요.
"""
    return prompt


patch_plan_prompt = build_patch_plan_prompt(game_overview, response_group_summary, patch_evidence_base)

print("패치/운영 제안 프롬프트 길이:", len(patch_plan_prompt))
print(patch_plan_prompt[:2000])

In [ ]:
# ============================================================
# 패치/운영 제안 Agent
# ============================================================
patch_plan_system_prompt = """
당신은 Steam 인디게임 출시 후 리뷰 분석 결과를 바탕으로 패치 및 운영 방향성을 정리하는 데이터 분석 보조자입니다.

역할:
- 이미 계산된 대응 구분과 집계값을 바탕으로 보고서용 제안 문장을 작성합니다.
- 우선순위를 새로 판단하지 않습니다.
- 수치를 새로 만들지 않습니다.
- 리뷰 근거가 없는 내용을 과장하지 않습니다.

작성 톤:
- 개발팀 회의에서 바로 사용할 수 있게 구체적으로 작성합니다.
- 단정적인 표현보다 "우선 확인", "검토", "개선 필요"처럼 데이터 기반 제안 톤을 유지합니다.
"""

if vertex_model is not None:
    patch_plan_agent = Agent(
        vertex_model,
        output_type=PostlaunchOperationPlan,
        system_prompt=patch_plan_system_prompt,
        retries=MAX_RETRIES,
        output_retries=3,
    )
else:
    patch_plan_agent = None

print("패치/운영 제안 Agent 생성 여부:", "O" if patch_plan_agent is not None else "X")

In [ ]:
# ============================================================
# 패치/운영 제안 LLM 실행
# ============================================================
if RUN_PATCH_PLAN_LLM:
    if patch_plan_agent is None:
        raise RuntimeError("patch_plan_agent가 생성되지 않았습니다. Vertex AI 설정을 확인하세요.")

    patch_plan_result = await patch_plan_agent.run(
        patch_plan_prompt,
        model_settings=patch_plan_settings,
    )
    patch_plan = patch_plan_result.output
else:
    patch_plan = None
    print("RUN_PATCH_PLAN_LLM=False이므로 LLM 호출 없이 프롬프트만 확인합니다.")

if patch_plan is not None:
    patch_plan_dict = to_serializable(patch_plan)

    with open(PATCH_PLAN_JSON_PATH, "w", encoding="utf-8") as f:
        json.dump(patch_plan_dict, f, ensure_ascii=False, indent=2)

    patch_tasks_df = pd.DataFrame(patch_plan_dict.get("patch_tasks", []))
    patch_tasks_df.to_csv(PATCH_PLAN_CSV_PATH, index=False, encoding="utf-8-sig")

    print("패치/운영 제안 JSON 저장:", PATCH_PLAN_JSON_PATH)
    print("패치/운영 작업 CSV 저장:", PATCH_PLAN_CSV_PATH)
    display(patch_tasks_df)

# 14. 보고서용 문장 저장 및 출력

In [ ]:
# ============================================================
# 보고서용 Markdown 생성
# ============================================================
if patch_plan is not None:
    patch_plan_dict = to_serializable(patch_plan)

    md_lines = []
    md_lines.append(f"# {TARGET_GAME_NAME} 출시 후 패치·운영 방향성 제안")
    md_lines.append("")
    md_lines.append("## 한 줄 요약")
    md_lines.append(patch_plan_dict.get("one_line_summary", ""))
    md_lines.append("")

    section_map = [
        ("지금 먼저 확인할 일", "check_now"),
        ("다음 업데이트에서 개선할 일", "next_update"),
        ("장기적으로 보강할 일", "long_term"),
        ("계속 살릴 강점", "keep_strength"),
    ]

    for title, key in section_map:
        md_lines.append(f"## {title}")
        for item in patch_plan_dict.get(key, []):
            md_lines.append(f"- {item}")
        md_lines.append("")

    md_lines.append("## 보고서용 설명")
    md_lines.append(patch_plan_dict.get("report_text", ""))
    md_lines.append("")

    md_text = "\n".join(md_lines)
    PATCH_PLAN_MD_PATH.write_text(md_text, encoding="utf-8")

    print("보고서용 Markdown 저장:", PATCH_PLAN_MD_PATH)
    display(Markdown(md_text))
else:
    print("patch_plan이 없습니다. RUN_PATCH_PLAN_LLM=True로 실행해야 보고서용 문장이 생성됩니다.")

# 15. 최종 산출물 점검

In [ ]:
# ============================================================
# 산출물 점검
# ============================================================
output_paths = {
    "LLM 입력 리뷰": LLM_INPUT_PATH,
    "필터 로그": FILTER_LOG_PATH,
    "샘플 요약": SAMPLE_SUMMARY_PATH,
    "리뷰 단위 LLM 결과 CSV": RESULT_CSV_PATH,
    "리뷰 단위 LLM 결과 JSON": RESULT_JSON_PATH,
    "이슈 태그 펼친 결과": ISSUE_TAG_FLAT_PATH,
    "게임 현황 요약": GAME_OVERVIEW_PATH,
    "이슈별 패치 요약": ISSUE_SUMMARY_PATH,
    "대응 유형별 요약": RESPONSE_GROUP_SUMMARY_PATH,
    "패치 제안 근거": PATCH_EVIDENCE_PATH,
    "Tableau 원천 데이터": TABLEAU_SOURCE_PATH,
    "패치 운영 제안 JSON": PATCH_PLAN_JSON_PATH,
    "패치 운영 작업 CSV": PATCH_PLAN_CSV_PATH,
    "보고서용 Markdown": PATCH_PLAN_MD_PATH,
}

check_rows = []
for name, path in output_paths.items():
    path = Path(path)
    rows = None
    cols = None
    if path.exists() and path.suffix.lower() == ".csv":
        try:
            temp = pd.read_csv(path)
            rows = len(temp)
            cols = len(temp.columns)
        except Exception:
            rows = "읽기 실패"
            cols = "읽기 실패"
    check_rows.append({
        "산출물": name,
        "exists": path.exists(),
        "rows": rows,
        "columns": cols,
        "path": str(path),
    })

output_check = pd.DataFrame(check_rows)
display(output_check)